# 02 — Knowledge Graph construction

This notebook turns the cleaned MSD × Lakh × jSymbolic dataframe into RDF triples that **extend** the project ontology in `notebooks/MusicRecSyst.ttl`. The original ontology file is never modified — every triple we mint is written to a sibling file, `notebooks/MusicRecSyst_populated.ttl`.

Pipeline:

1. **Load + clean** the per-track parquet, inner-join with the jSymbolic interim features, derive `tempo_class`.
2. **Restrict** the user taste profile to KG-covered songs.
3. **Populate** the ontology through `KGBuilder` (artists, tracks, performances, tempo classes, genres, instruments, jSymbolic numeric features). Genre and instrument leaves are dual-typed `mrc:Genre`/`mo:Instrument` **and** `skos:Concept`, ready for the SKOS enrichment below.
4. **Enrich** instruments and genres against Wikidata using the **SKOS** model (`skos:Concept`, `skos:inScheme`, `skos:exactMatch`, `skos:broader`, `skos:prefLabel`/`altLabel`/`definition`). All Wikidata calls are parallelised; English label, description, and aliases are pulled in one batched `wbgetentities` round.
5. **Add a temporal axis** with a `mrc:scheme/Decades` SKOS scheme — every track gets `mrc:inDecade`, decades are sequenced via `wdt:P155` (follows) / `wdt:P156` (followed by), and each is matched to its Wikidata decade entity (Q39825) and the parent century (`wdt:P361`).

All helpers live in `src/data/kg/`.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# notebooks/ → project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR          = PROJECT_ROOT / "data"
PARQUET_PATH      = DATA_DIR / "processed" / "lakh_msd_dataset.parquet"
INTERIM_CSV       = DATA_DIR / "interim"   / "interim.csv"
KG_INPUT_PQ       = DATA_DIR / "interim"   / "kg_input.parquet"          # cached, cleaned join
TASTE_FILTERED_PQ = DATA_DIR / "processed" / "taste_profile_filtered.parquet"  # full filtered profile (≥5 plays)
KG_TASTE_PQ       = DATA_DIR / "interim"   / "kg_taste_profile.parquet"  # restricted to KG songs, re-filtered
ONTO_BASE         = PROJECT_ROOT / "notebooks" / "MusicRecSyst.ttl"
ONTO_OUT          = PROJECT_ROOT / "notebooks" / "MusicRecSyst_populated.ttl"

# Set to True to ignore the cached parquet / TTL and rebuild from scratch.
FORCE_REBUILD = False

print("PROJECT_ROOT       :", PROJECT_ROOT)
print("parquet            :", PARQUET_PATH.exists(),      PARQUET_PATH)
print("interim csv        :", INTERIM_CSV.exists(),       INTERIM_CSV)
print("kg input pq        :", KG_INPUT_PQ.exists(),       KG_INPUT_PQ)
print("taste filtered pq  :", TASTE_FILTERED_PQ.exists(), TASTE_FILTERED_PQ)
print("kg taste pq        :", KG_TASTE_PQ.exists(),       KG_TASTE_PQ)
print("ontology base      :", ONTO_BASE.exists(),         ONTO_BASE)


PROJECT_ROOT       : /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project
parquet            : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/processed/lakh_msd_dataset.parquet
interim csv        : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/interim.csv
kg input pq        : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/kg_input.parquet
taste filtered pq  : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/processed/taste_profile_filtered.parquet
kg taste pq        : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/kg_taste_profile.parquet
ontology base      : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/notebooks/MusicRecSyst.ttl


In [2]:
# Force-reload the data.kg subpackage so notebook iterations pick up code edits
# without needing a full kernel restart.
import importlib, sys
for _name in [m for m in sys.modules if m == "data.kg" or m.startswith("data.kg.")]:
    importlib.reload(sys.modules[_name])


## 1 · Build (or load) the cleaned KG input

We want a single, reproducible dataframe that drives the KG build. The pipeline is:

1. Load the per-track parquet, project to `DEFAULT_KG_COLUMNS`.
2. Load `interim.csv` (jSymbolic features) and join on the **MIDI md5**
   (extracted from the Windows-style path stem).
3. **Drop** every row that has no jSymbolic features attached — only ~37 % of
   the parquet rows have a matching jSymbolic extraction (`7,113 / 19,137`),
   and the KG entries for the rest would be strictly poorer than what we
   already store on disk. We choose to keep *only* tracks that have both
   MSD metadata **and** jSymbolic features, so every node in the graph carries
   the full feature stack.
4. Apply the rename map (`KG_RENAME_MAP`): `release → album_name`,
   `key_name → key`, `mode_name → mode`. The integer-coded `key`/`mode`
   columns from the MSD parquet are *not* selected — we keep only the
   human-readable spellings.
5. Derive the categorical `tempo_class` from `Mean_Tempo` (Music Theory Academy
   nine-class scheme).
6. **Cache** the result to `data/interim/kg_input.parquet`.

On subsequent runs (when `FORCE_REBUILD=False` and the cache exists) we just
load the parquet — no re-joining, no re-cleaning.

Columns kept from the parquet (`DEFAULT_KG_COLUMNS` in
`src/data/kg/variable_selection.py`), shown under their **post-rename** names:

* identifiers — `track_id`, `midi_path`, `song_id`, `artist_id`, `artist_mbid`
* labels      — `artist_name`, `title`, `album_name` *(was `release`)*, `year`
* tonality    — `key` *(was `key_name`)*, `key_confidence`,
                `mode` *(was `mode_name`)*, `mode_confidence`
* MSD audio   — `time_signature`, `time_signature_confidence`, `loudness`
* tags        — `primary_genre`, `top3_genres`, `artist_terms`
* MIDI        — `midi_n_instruments`, `midi_instrument_names`

On top of these we attach all `INTERIM_KG_FEATURES` (jSymbolic numerics:
`Mean_Tempo`, `Number_of_Pitches`, `Pitch_Variety`, etc.) via the inner-join,
and finally the derived `tempo_class` column.


In [3]:
import pandas as pd

from data.kg import (
    DEFAULT_KG_COLUMNS,
    INTERIM_KG_FEATURES,
    select_kg_columns,
    load_interim_features,
    merge_parquet_with_interim,
)
from data.kg.tempo_classes import add_tempo_class_column


def build_kg_input() -> pd.DataFrame:
    """Project parquet → join with interim → drop unmatched → add tempo_class."""
    raw   = pd.read_parquet(PARQUET_PATH)
    print(f"  raw parquet           : {raw.shape}")

    kg_df = select_kg_columns(raw)
    print(f"  projected to KG cols  : {kg_df.shape}")

    interim_df = load_interim_features(INTERIM_CSV, INTERIM_KG_FEATURES)
    print(f"  interim features      : {interim_df.shape}")

    # inner-join: keep only tracks that have BOTH MSD metadata AND jSymbolic
    merged = merge_parquet_with_interim(
        kg_df, interim_df, midi_path_col="midi_path", how="inner",
    )
    print(f"  inner-joined          : {merged.shape}")

    # belt-and-braces: drop any row that ended up without a Mean_Tempo
    before = len(merged)
    merged = merged.dropna(subset=["Mean_Tempo"]).reset_index(drop=True)
    if len(merged) != before:
        print(f"  dropped {before - len(merged):,} rows w/o Mean_Tempo")

    merged = add_tempo_class_column(merged, tempo_col="Mean_Tempo",
                                    out_col="tempo_class")
    return merged


if KG_INPUT_PQ.exists() and not FORCE_REBUILD:
    print(f"loading cached KG input from {KG_INPUT_PQ}")
    merged = pd.read_parquet(KG_INPUT_PQ)
    # tempo_class round-trips as plain object/category — rebuild the ordered Categorical
    merged = add_tempo_class_column(merged, tempo_col="Mean_Tempo",
                                    out_col="tempo_class")
else:
    print("building KG input from scratch …")
    merged = build_kg_input()
    KG_INPUT_PQ.parent.mkdir(parents=True, exist_ok=True)
    # Categorical doesn't survive parquet round-trip cleanly with dtype=category
    # in some pyarrow versions, so cast to plain string before writing.
    out = merged.copy()
    out["tempo_class"] = out["tempo_class"].astype("string")
    out.to_parquet(KG_INPUT_PQ, index=False)
    print(f"  cached → {KG_INPUT_PQ} ({KG_INPUT_PQ.stat().st_size/1024:,.1f} KiB)")

print(f"\nfinal KG input: {merged.shape}")
merged.head(3)


loading cached KG input from /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/kg_input.parquet

final KG input: (7113, 55)


/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/SONATAM/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,track_id,midi_path,song_id,artist_id,artist_mbid,artist_name,title,album_name,key,key_confidence,...,Acoustic_Guitar_Prevalence,Electric_Guitar_Prevalence,Brass_Prevalence,Woodwinds_Prevalence,Average_Number_of_Independent_Voices,Variability_of_Number_of_Independent_Voices,Voice_Equality_-_Number_of_Notes,Variation_of_Dynamics,Average_Note_to_Note_Change_in_Dynamics,tempo_class
0,TRAAAGR128F425B14B,/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJ...,SONRWUU12AF72A4283,ARGE7G11187FB37E05,7bd9e20e-74b9-446a-a2ed-a223f82a36e7,Cyndi Lauper,Into The Nightlife,Bring Ya To The Brink,A,0.608,...,0.0,0.12870,0.01361,0.01717,3.766,2.152,557.70,26.410,13.070,Allegro
1,TRAABVM128F92CA9DC,/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJ...,SOXLBJT12A8C140925,ARYKCQI1187FB3B18F,eeacb319-8d4c-48e0-80a0-944e71c375bf,Tesla,Caught In A Dream,Gold,G,0.725,...,0.0,0.17370,0.00000,0.00000,4.001,1.562,418.80,19.690,13.590,Adagio
2,TRAADKW128E079503A,/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJ...,SOJCRUY12A67ADA4C2,ARAO91X1187B98CCA4,1129817c-488a-4096-80c1-77fc1b107c93,Tracy Chapman,Fast Car (LP Version),Tracy Chapman,D,0.018,...,0.3,0.08182,0.00000,0.22610,5.859,1.658,65.25,9.977,3.734,Andante


## 2 · `tempo_class` reference + distribution

We use the [Music Theory Academy](https://www.musictheoryacademy.com/understanding-music/tempo/) nine-class scheme (Larghissimo … Prestissimo). The classifier reads from `Mean_Tempo`; the table below is rendered straight from `tempo_class_table()`.


In [4]:
from data.kg.tempo_classes import tempo_class_table

print(f"tempo_class distribution (n = {len(merged):,}):")
print(merged["tempo_class"].value_counts(dropna=False).sort_index())

tempo_class_table()


tempo_class distribution (n = 7,113):
tempo_class
Larghissimo       0
Grave            15
Largo           109
Adagio          853
Andante        2322
Moderato        844
Allegro        2611
Presto          279
Prestissimo      80
Name: count, dtype: int64


,tempo_class,min_bpm,max_bpm,description
0,Larghissimo,0.0,20.0,Extremely slow (< 20 BPM)
1,Grave,20.0,40.0,Slow and solemn (20–40 BPM)
2,Largo,40.0,60.0,Very slow (40–60 BPM)
3,Adagio,60.0,76.0,Slow and stately (60–76 BPM)
4,Andante,76.0,108.0,Walking pace (76–108 BPM)
5,Moderato,108.0,120.0,Moderately (108–120 BPM)
6,Allegro,120.0,168.0,Fast (120–168 BPM)
7,Presto,168.0,200.0,Very fast (168–200 BPM)
8,Prestissimo,200.0,inf,Very very fast (≥ 200 BPM)


## 3 · Restrict the taste profile to KG songs

`taste_profile_filtered.parquet` (built in `00b_user_data_integration.ipynb`)
contains every `(user_id, song_id, play_count)` interaction surviving the
`MIN_PLAYS_PER_USER = 5` cold-start filter, **at the time it was built**.

But the KG only covers the 7,113 tracks that have *both* MSD metadata and
jSymbolic features — so any song outside that set effectively disappears from
our recommendation universe. Once we drop those interactions some users may
fall below the 5-play threshold, so we **re-apply the filter** to keep the
distribution honest.

The helper `load_or_build_kg_taste_profile`:

1. loads `taste_profile_filtered.parquet`,
2. keeps only rows whose `song_id` is in the KG (`merged["song_id"]`),
3. re-applies `min_plays_per_user=5`,
4. caches the result to `data/interim/kg_taste_profile.parquet`.

Subsequent runs just reload the cache (unless `FORCE_REBUILD=True`).


In [5]:
from data.kg import load_or_build_kg_taste_profile

kg_song_ids = set(merged["song_id"].dropna().unique())
print(f"KG covers {len(kg_song_ids):,} unique song_ids")

taste = load_or_build_kg_taste_profile(
    cache_path=KG_TASTE_PQ,
    taste_source=TASTE_FILTERED_PQ,
    kg_song_ids=kg_song_ids,
    min_plays_per_user=5,
    force_rebuild=FORCE_REBUILD,
)

print(f"\nKG-restricted taste profile: {taste.shape}")
print(f"  unique users : {taste['user_id'].nunique():,}")
print(f"  unique songs : {taste['song_id'].nunique():,}")
print(f"  interactions : {len(taste):,}")
print("\nplays per user (post-filter):")
print(taste.groupby("user_id").size().describe())


KG covers 7,013 unique song_ids
[user_data] loading cache from /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/kg_taste_profile.parquet

KG-restricted taste profile: (2853630, 3)
  unique users : 285,037
  unique songs : 7,010
  interactions : 2,853,630

plays per user (post-filter):

KG-restricted taste profile: (2853630, 3)
  unique users : 285,037
  unique songs : 7,010
  interactions : 2,853,630

plays per user (post-filter):
count    285037.000000
mean         10.011437
std           6.934456
min           5.000000
25%           6.000000
50%           8.000000
75%          12.000000
max         197.000000
dtype: float64
count    285037.000000
mean         10.011437
std           6.934456
min           5.000000
25%           6.000000
50%           8.000000
75%          12.000000
max         197.000000
dtype: float64


## 4 · Populate the ontology

`KGBuilder` opens a fresh copy of `MusicRecSyst.ttl` at `MusicRecSyst_populated.ttl` and adds new triples on top of it. For each row it mints (idempotently, by URI):

* `mo:MusicArtist` keyed by `artist_id` / `artist_mbid`
* `mrc:MSDTrack` keyed by `track_id`, with `dcterms:title` / `dcterms:date` / release info
* `mo:Performance` linking artist ↔ track, carrying `mo:tempo`, `mo:key`, `mrc:hasTempoClass`, and one `mo:instrument` per GM name
* `mrc:Genre` and `mo:Instrument` leaves — **also** typed `skos:Concept` and placed in `mrc:scheme/Genres` / `mrc:scheme/Instruments` so the Wikidata enrichment in §6 can attach `skos:exactMatch` and `skos:broader` on top
* the controlled vocabulary of `mrc:TempoClass` individuals
* one `mrc:*` datatype property per jSymbolic numeric feature

If `MusicRecSyst_populated.ttl` already exists and `FORCE_REBUILD` is `False` we just load it back; otherwise we rebuild from scratch.


In [6]:
from data.kg import KGBuilder

if ONTO_OUT.exists() and not FORCE_REBUILD:
    print(f"loading cached populated graph from {ONTO_OUT}")
    # Re-instantiating with overwrite_copy=False reuses the existing TTL
    builder = KGBuilder(
        base_ttl=ONTO_BASE,
        out_ttl=ONTO_OUT,
        overwrite_copy=False,
    )
    print("graph stats:", builder.stats())
else:
    print("building knowledge graph from scratch …")
    builder = KGBuilder(
        base_ttl=ONTO_BASE,
        out_ttl=ONTO_OUT,
        overwrite_copy=True,
    )
    builder.add_tempo_class_individuals()

    counts = builder.populate_from_dataframe(merged, max_rows=None, verbose=True)
    print("\nminted (this run):", counts)
    print("graph stats      :", builder.stats())

    builder.save()
    print(f"\nwrote {ONTO_OUT}  ({ONTO_OUT.stat().st_size / 1024:,.1f} KiB)")


loading cached populated graph from /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/notebooks/MusicRecSyst_populated.ttl


KeyboardInterrupt: 

## 5 · Inspect

A couple of SPARQL sanity checks against the in-memory graph.


In [ ]:
total_tracks = builder.stats()["tracks"]

q_tracks_per_class = """
PREFIX mrc:  <http://purl.org/ontology/mrc/>
PREFIX mo:   <http://purl.org/ontology/mo/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?className (COUNT(DISTINCT ?perf) AS ?n)
WHERE {
    ?perf a mo:Performance ;
          mrc:hasTempoClass ?tc .
    ?tc rdfs:label ?className .
}
GROUP BY ?className
ORDER BY DESC(?n)
"""
print("performances per tempo class:")
count = 0
for row in builder.g.query(q_tracks_per_class):
    label_count = int(row[1])
    print(f"  {str(row[0]):<14} {label_count:>5} ({label_count/total_tracks:.1%})")
    count += label_count

performances per tempo class:
  Allegro         2573 (36.7%)
  Andante         2293 (32.7%)
  Adagio           840 (12.0%)
  Moderato         834 (11.9%)
  Presto           273 (3.9%)
  Largo            108 (1.5%)
  Prestissimo       79 (1.1%)
  Grave             13 (0.2%)
  Allegro         2573 (36.7%)
  Andante         2293 (32.7%)
  Adagio           840 (12.0%)
  Moderato         834 (11.9%)
  Presto           273 (3.9%)
  Largo            108 (1.5%)
  Prestissimo       79 (1.1%)
  Grave             13 (0.2%)


In [ ]:
#TODO CHANGE TO BE SPECIFIC FOR ARTISTS

q_top_genres = """
PREFIX mrc:  <http://purl.org/ontology/mrc/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?label (COUNT(?a) AS ?n)
WHERE {
    ?a mrc:hasGenre ?g .
    ?g skos:prefLabel ?label .
}
GROUP BY ?label
ORDER BY DESC(?n)
LIMIT 10
"""
print("\ntop 10 genres by artist count:")
total_artists = builder.stats()["artists"]
for row in builder.g.query(q_top_genres):
    label_count = int(row[1])
    print(f"  {str(row[0]):<30} {label_count:>5} ({label_count/total_artists:.1%})")


top 10 genres by artist count:
  rock                            2117 (57.7%)
  pop                             1062 (29.0%)
  electronic                      1037 (28.3%)
  pop rock                         814 (22.2%)
  hip hop                          381 (10.4%)
  jazz                             349 (9.5%)
  alternative rock                 337 (9.2%)
  indie rock                       290 (7.9%)
  disco                            282 (7.7%)
  soft rock                        277 (7.6%)
  rock                            2117 (57.7%)
  pop                             1062 (29.0%)
  electronic                      1037 (28.3%)
  pop rock                         814 (22.2%)
  hip hop                          381 (10.4%)
  jazz                             349 (9.5%)
  alternative rock                 337 (9.2%)
  indie rock                       290 (7.9%)
  disco                            282 (7.7%)
  soft rock                        277 (7.6%)


## 6 · Wikidata enrichment — SKOS edition

Local instrument and genre nodes start as flat leaves with a single label. We link them to Wikidata using the **SKOS** data model (industry standard for thesauri / controlled vocabularies):

* `skos:exactMatch` from each local concept to its Wikidata QID — *not* `owl:sameAs`, because identity between OWL individuals is too strong a claim for a thesaurus alignment.
* every QID node is itself a `skos:Concept` placed in the same `skos:ConceptScheme` (`mrc:scheme/Genres` / `mrc:scheme/Instruments`).
* hierarchy uses `skos:broader` between QID nodes (no `rdfs:subClassOf` — these are concepts, not classes of works).
* every QID node carries `skos:prefLabel` (canonical English label), `skos:definition` (English description), and one `skos:altLabel` per English alias — pulled in one batched `wbgetentities` call.

**Performance.** Three Wikidata endpoints are used and **all are parallelised** through `ThreadPoolExecutor`:

| endpoint | what for | default workers |
|---|---|---|
| WDQS SPARQL | exact-label resolution + bounded `wdt:P279*` ancestor walk | 6 |
| `wbsearchentities` | fuzzy label fallback for stubborn names | 6 |
| `wbgetentities` (≤50 IDs/call, batched) | English label / description / aliases for every minted QID | 4 |

Caches in `data/interim/wikidata_*.json` — second run is O(1).


In [ ]:
RUN_WIKIDATA = True   # set False to skip the network-bound enrichment

if RUN_WIKIDATA:
    from data.kg import (
        INSTRUMENT_ROOT, GENRE_ROOT,
        resolve_labels,
    )
    from data.kg.kg_builder import _iter_strings

    WD_INSTR_PQ        = DATA_DIR / "interim" / "wikidata_instruments.json"
    WD_INSTR_CHAINS_PQ = DATA_DIR / "interim" / "wikidata_instrument_chains.json"
    WD_GENRE_PQ        = DATA_DIR / "interim" / "wikidata_genres.json"
    WD_GENRE_CHAINS_PQ = DATA_DIR / "interim" / "wikidata_genre_chains.json"
    WD_QID_META_PQ     = DATA_DIR / "interim" / "wikidata_qid_metadata.json"

    # ── 6.1 collect unique labels actually used in the KG ──────────────────
    instr_labels: set[str] = set()
    for cell in merged["midi_instrument_names"]:
        instr_labels.update(_iter_strings(cell))

    genre_labels: set[str] = set()
    for cell in merged["primary_genre"]:
        if isinstance(cell, str) and cell.strip():
            genre_labels.add(cell.strip())
    for col in ("top3_genres", "artist_terms"):
        for cell in merged[col]:
            genre_labels.update(_iter_strings(cell))

    print(f"unique instrument labels : {len(instr_labels):,}")
    print(f"unique genre labels      : {len(genre_labels):,}")

    # ── 6.2 resolve labels → QIDs (parallel, cached) ───────────────────────
    instr_map = resolve_labels(
        sorted(instr_labels), INSTRUMENT_ROOT, cache_path=WD_INSTR_PQ,
        force_refresh=FORCE_REBUILD, parallel=6,
    )
    genre_map = resolve_labels(
        sorted(genre_labels), GENRE_ROOT, cache_path=WD_GENRE_PQ,
        force_refresh=FORCE_REBUILD, parallel=6,
    )
    print(f"\ninstrument hits: {sum(1 for v in instr_map.values() if v):,}"
          f" / {len(instr_map):,}")
    print(f"genre hits     : {sum(1 for v in genre_map.values() if v):,}"
          f" / {len(genre_map):,}")


unique instrument labels : 129
unique genre labels      : 645
[wikidata] resolving 129 new labels (type_root=Q110295396, workers=6) ...


labels: 100%|██████████| 129/129 [00:28<00:00,  4.45item/s]


[wikidata] cache -> /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/wikidata_instruments.json
[wikidata] resolving 645 new labels (type_root=Q188451, workers=6) ...


labels:  44%|████▍     | 286/645 [02:26<03:03,  1.96item/s]



In [ ]:
if RUN_WIKIDATA:
    from data.kg import (
        fetch_subclass_chains,
        fetch_qid_metadata,
        enrich_graph_with_wikidata,
    )

    # ── 6.3 expand bounded P279* chains for the resolved QIDs (parallel) ───
    instr_qids = [q for q in instr_map.values() if q]
    genre_qids = [q for q in genre_map.values() if q]

    instr_chains = fetch_subclass_chains(
        instr_qids, INSTRUMENT_ROOT, cache_path=WD_INSTR_CHAINS_PQ,
        force_refresh=FORCE_REBUILD, parallel=6,
    )
    genre_chains = fetch_subclass_chains(
        genre_qids, GENRE_ROOT, cache_path=WD_GENRE_CHAINS_PQ,
        force_refresh=FORCE_REBUILD, parallel=6,
    )

    # ── 6.4 fetch English label / description / aliases for every QID we'll
    #         mint into the graph (batched wbgetentities, parallel)
    all_qids: set[str] = set(instr_qids) | set(genre_qids)
    for chain in (*instr_chains.values(), *genre_chains.values()):
        for c_qid, _ in chain:
            all_qids.add(c_qid)

    qid_metadata = fetch_qid_metadata(
        sorted(all_qids), cache_path=WD_QID_META_PQ,
        force_refresh=FORCE_REBUILD, parallel=4, batch_size=50,
    )

    # ── 6.5 fold into the populated KG with the SKOS data model ────────────
    enrich_counts = enrich_graph_with_wikidata(
        builder,
        instrument_map=instr_map,
        genre_map=genre_map,
        instrument_chains=instr_chains,
        genre_chains=genre_chains,
        qid_metadata=qid_metadata,
        add_hierarchy=True,
    )
    print("\nenrichment summary :", enrich_counts)
    print("graph stats (after):", builder.stats())

    builder.save()
    print(f"\nupdated {ONTO_OUT}  ({ONTO_OUT.stat().st_size / 1024:,.1f} KiB)")


[wikidata] loaded 45 cached subclass chains from wikidata_instrument_chains.json
[wikidata] expanding 0 new QIDs (root=Q110295396, workers=6) ...
[wikidata] cache -> /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/wikidata_instrument_chains.json
[wikidata] loaded 426 cached subclass chains from wikidata_genre_chains.json
[wikidata] expanding 0 new QIDs (root=Q188451, workers=6) ...
[wikidata] cache -> /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/wikidata_genre_chains.json
[wikidata] loaded 723 cached QID-metadata entries from wikidata_qid_metadata.json
[wikidata] fetching metadata for 0 new QIDs (workers=4, batch=50) ...
[wikidata] enrichment summary: {'instrument_links': 66, 'genre_links': 478, 'qid_concepts': 712, 'broader_edges': 2616, 'subclass_edges': 3160}

enrichment summary : {'instrument_links': 66, 'genre_links': 478, 'qid_concepts': 712, 'broader_edges': 2616, 'subclass_edges': 3160}
graph stats (after): {'triples'

### 6.5b · Audit — what did Wikidata actually add?

Two follow-up questions worth answering directly:

1. **Did the enrichment add nodes for instruments / genres that are *not*
   in the MIDIs?**  Yes, but only as *ancestor* classes (e.g. *string
   instrument*, *keyboard instrument*) — the leaves are always the labels
   we extracted from the dataset. The audit below splits every Wikidata
   node into `wd_anchors` (= a leaf's `skos:exactMatch` target — appears
   in your data) vs. `wd_ancestors_only` (= added purely to give a leaf
   a parent — never used as a leaf).
2. **Why does Protégé show genres / instruments as a flat list?**
   Because Protégé's *Class hierarchy* tab only renders `rdfs:subClassOf`
   between `owl:Class` nodes — `skos:broader` between `skos:Concept`
   *individuals* renders nowhere unless you install the SKOS plugin.
   `enrich_graph_with_wikidata(..., protege_friendly=True)` (the default)
   now dual-emits the hierarchy: every Wikidata anchor is *also* typed
   `owl:Class`, every `skos:broader` is mirrored as `rdfs:subClassOf`,
   and every local genre/instrument is rooted under `mrc:Genre` /
   `mo:Instrument`. Reload the populated TTL in Protégé → expand
   `mo:Instrument` and `mrc:Genre` in the *Classes* tab and you will see
   the full Wikidata-derived tree.


In [ ]:
if RUN_WIKIDATA:
    from data.kg import audit_wikidata_enrichment

    audit = audit_wikidata_enrichment(
        builder,
        instrument_map=instr_map,
        genre_map=genre_map,
        show_examples=8,
        verbose=True,
    )

    # One-line headline per scheme: how much of the graph is "extra" hierarchy?
    for name, r in audit.items():
        in_data   = r["wd_anchors"]
        ancestors = r["wd_ancestors_only"]
        total_wd  = in_data + ancestors
        ratio     = (ancestors / total_wd) if total_wd else 0.0
        print(f"\n{name}: {ancestors:,} ancestor-only Wikidata nodes "
              f"({ratio:.0%} of all WD nodes in scheme) — added solely as "
              f"parents of the {in_data:,} dataset matches.")

    # Sanity: confirm Protégé will see a class hierarchy.
    q = """
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT
      (COUNT(DISTINCT ?c) AS ?n_class)
      (COUNT(DISTINCT ?s) AS ?n_sub_subj)
    WHERE {
      { ?c a owl:Class .
        FILTER(STRSTARTS(STR(?c), "http://www.wikidata.org/")) }
      UNION
      { ?s rdfs:subClassOf ?o .
        FILTER(STRSTARTS(STR(?s), "http://www.wikidata.org/") ||
               STRSTARTS(STR(?o), "http://www.wikidata.org/")) }
    }
    """
    for row in builder.g.query(q):
        print(f"\nProtégé Class-tab visibility: "
              f"{int(row.n_class):,} Wikidata owl:Class nodes, "
              f"{int(row.n_sub_subj):,} rdfs:subClassOf subjects "
              f"touching Wikidata.")



── INSTRUMENTS ──
  leaves_total           : 129
  leaves_linked          : 66
  leaves_orphan          : 63
  wd_anchors             : 48
  wd_ancestors_only      : 112
  broader_edges          : 435
  subclass_edges         : 487
  ancestor_examples      :
      Q336767    squeezebox
      Q1808578   lute
      Q55721729  percussion vessel
      Q19589438  simple chordophone
      Q1606685   vessel drum
      Q11853199  vessel flute
      Q98329515  continuous-pitch instrument
      Q54819588  interruptive free aerophone
  raw_labels_total       : 129
  raw_labels_hit         : 66
  raw_labels_missed      : 63

── GENRES ──
  leaves_total           : 645
  leaves_linked          : 489
  leaves_orphan          : 156
  wd_anchors             : 449
  wd_ancestors_only      : 130
  broader_edges          : 2037
  subclass_edges         : 2441
  ancestor_examples      :
      Q108857328 music of Eastern Europe
      Q1373485   underground music
      Q5645984   music of South America
   

### 6.6 · Sanity-check the SKOS hierarchy

Two SPARQL probes:

1. how many local genre/instrument leaves got a `skos:exactMatch` to a Wikidata QID;
2. for `piano`, walk `skos:broader*` from the local instrument node up through the Wikidata chain — should give *piano → struck-string instrument → string instrument → musical instrument*, each row carrying its `skos:prefLabel` / `skos:definition` from Wikidata.


In [ ]:
if RUN_WIKIDATA:
    q_link_counts = """
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX mo:   <http://purl.org/ontology/mo/>
    PREFIX mrc:  <http://purl.org/ontology/mrc/>
    SELECT
        (COUNT(DISTINCT ?gi) AS ?genres_linked)
        (COUNT(DISTINCT ?ii) AS ?instruments_linked)
    WHERE {
      OPTIONAL { ?gi a mrc:Genre     ; skos:exactMatch ?gw .
                 FILTER(STRSTARTS(STR(?gw), "http://www.wikidata.org/")) }
      OPTIONAL { ?ii a mo:Instrument ; skos:exactMatch ?iw .
                 FILTER(STRSTARTS(STR(?iw), "http://www.wikidata.org/")) }
    }
    """
    for row in builder.g.query(q_link_counts):
        print(f"genres linked      : {int(row.genres_linked):>4}")
        print(f"instruments linked : {int(row.instruments_linked):>4}")

    # Walk piano-family hierarchy: local node → skos:exactMatch → wd → skos:broader*
    q_piano_chain = """
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX mo:   <http://purl.org/ontology/mo/>
    SELECT DISTINCT ?local_label ?ancestor ?ancestor_label ?ancestor_def
    WHERE {
      ?local a mo:Instrument ;
             skos:prefLabel  ?local_label ;
             skos:exactMatch ?wd .
      FILTER(CONTAINS(LCASE(STR(?local_label)), "piano"))
      ?wd skos:broader* ?ancestor .
      OPTIONAL { ?ancestor skos:prefLabel ?ancestor_label .
                 FILTER(LANG(?ancestor_label) = "en") }
      OPTIONAL { ?ancestor skos:definition ?ancestor_def .
                 FILTER(LANG(?ancestor_def) = "en") }
    }
    ORDER BY ?local_label ?ancestor
    """
    print("\npiano-family instruments → Wikidata ancestors :")
    last_local = None
    for row in builder.g.query(q_piano_chain):
        local = str(row.local_label)
        if local != last_local:
            print(f"\n  {local}")
            last_local = local
        qid = str(row.ancestor).rsplit("/", 1)[-1]
        defn = str(row.ancestor_def)[:60] + "…" if row.ancestor_def else ""
        print(f"     {qid:10s} {str(row.ancestor_label or ''):<32} {defn}")


genres linked      :  489
instruments linked :   66

piano-family instruments → Wikidata ancestors :

  Acoustic Grand Piano
     Q34379     musical instrument               device created or adapted to make musical sounds…
     Q52954     keyboard instrument              class of musical instrument which is played using a musical …
     Q5994      piano                            musical keyboard instrument…

  Bright Acoustic Piano
     Q34379     musical instrument               device created or adapted to make musical sounds…
     Q52954     keyboard instrument              class of musical instrument which is played using a musical …
     Q5994      piano                            musical keyboard instrument…

  Electric Grand Piano
     Q34379     musical instrument               device created or adapted to make musical sounds…
     Q52954     keyboard instrument              class of musical instrument which is played using a musical …
     Q5994      piano                   

## 7 · Temporal axis — decades

We add a `mrc:scheme/Decades` SKOS scheme so the recommender can reason about *time* the same way it reasons about genre and instrument. For every distinct release decade in the data we mint:

* a local node (e.g. `ex:decade/1960s`) typed `mrc:Decade` + `skos:Concept`, with `mrc:startYear` / `mrc:endYear` (xsd:gYear) and `skos:prefLabel "1960s"@en`;
* a local sequence using `wdt:P155` (follows) / `wdt:P156` (followed by) so a SPARQL walk can iterate decades in order;
* `skos:exactMatch` to the Wikidata decade entity (resolved via `wdt:P31 wd:Q39825` + `YEAR(wdt:P580)`), plus `wdt:P361` to the parent century and `skos:broader` to the same century — exactly the pattern in your example (`wd:Q36297 wdt:P31 wd:Q39825 ; wdt:P361 wd:Q6927 ; wdt:P155 wd:Q35024`).

Every track is then attached with `mrc:inDecade <ex:decade/…>`. Wikidata calls reuse the parallel session pool from §6 and are cached to `data/interim/wikidata_decades.json`; QID metadata (label/description/aliases for decade *and* century entities) is fetched in the same batched `wbgetentities` round.


In [ ]:
from data.kg import (
    unique_decades_from_dataframe,
    resolve_decades,
    collect_decade_qids_for_metadata,
    add_decades_to_graph,
    fetch_qid_metadata,
)

WD_DECADES_PQ  = DATA_DIR / "interim" / "wikidata_decades.json"
WD_QID_META_PQ = DATA_DIR / "interim" / "wikidata_qid_metadata.json"

# 7.1 — distinct decade starts present in the data
decade_starts = unique_decades_from_dataframe(merged, year_col="year")
print(f"distinct decades in data : {len(decade_starts)}")
print(f"  range                  : {decade_starts[0]}s … {decade_starts[-1]}s")

# 7.2 — resolve each decade to its Wikidata QID + parent century (parallel)
decade_qids = resolve_decades(
    decade_starts, cache_path=WD_DECADES_PQ,
    force_refresh=FORCE_REBUILD, parallel=4,
)
hits = sum(1 for v in decade_qids.values() if v and v.get("qid"))
print(f"\ndecades resolved on Wikidata : {hits} / {len(decade_qids)}")

# 7.3 — pull English label/description/aliases for the decade + century QIDs.
#       Merge into the existing qid_metadata cache so SPARQL prefLabel lookups
#       work uniformly for instruments, genres, AND decades.
decade_meta_qids = collect_decade_qids_for_metadata(decade_qids)
qid_metadata = fetch_qid_metadata(
    decade_meta_qids, cache_path=WD_QID_META_PQ,
    force_refresh=FORCE_REBUILD, parallel=4, batch_size=50,
)
print(f"qid metadata entries (cumulative) : {len(qid_metadata):,}")


distinct decades in data : 7
  range                  : 1950s … 2010s
[decades] loaded 7 cached decade entries from wikidata_decades.json
[decades] resolving 0 new decades (workers=4) ...
[decades] cache -> /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/wikidata_decades.json

decades resolved on Wikidata : 7 / 7
[wikidata] loaded 723 cached QID-metadata entries from wikidata_qid_metadata.json
[wikidata] fetching metadata for 0 new QIDs (workers=4, batch=50) ...
qid metadata entries (cumulative) : 723
[decades] cache -> /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/wikidata_decades.json

decades resolved on Wikidata : 7 / 7
[wikidata] loaded 723 cached QID-metadata entries from wikidata_qid_metadata.json
[wikidata] fetching metadata for 0 new QIDs (workers=4, batch=50) ...
qid metadata entries (cumulative) : 723


In [ ]:
# 7.4 — fold decades into the KG, attach every track via mrc:inDecade,
#        sequence them with wdt:P155 / wdt:P156, link to centuries.
decade_counts = add_decades_to_graph(
    builder, merged,
    year_col="year", track_id_col="track_id",
    decade_qids=decade_qids,
    qid_metadata=qid_metadata,
    verbose=True,
)
print("\ndecade enrichment :", decade_counts)
print("graph stats       :", builder.stats())

builder.save()
print(f"\nupdated {ONTO_OUT}  ({ONTO_OUT.stat().st_size / 1024:,.1f} KiB)")


[decades] summary: {'decades': 7, 'track_links': 4834, 'qid_concepts': 11}

decade enrichment : {'decades': 7, 'track_links': 4834, 'qid_concepts': 11}
graph stats       : {'triples': 281946, 'artists': 3668, 'tracks': 7013, 'performances': 7013, 'genres': 1039, 'instruments': 170, 'tempo_classes': 9, 'decades': 7, 'skos_concepts': 1534}

updated /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/notebooks/MusicRecSyst_populated.ttl  (14,882.6 KiB)

updated /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/notebooks/MusicRecSyst_populated.ttl  (14,882.6 KiB)


In [ ]:
# 7.5 — sanity-check: tracks per decade, walking the local sequence in order.
q_decade_walk = """
PREFIX mrc:  <http://purl.org/ontology/mrc/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
SELECT ?label ?start (COUNT(DISTINCT ?t) AS ?n_tracks) ?wd
WHERE {
    ?d a mrc:Decade ;
       skos:prefLabel ?label ;
       mrc:startYear  ?start .
    OPTIONAL { ?d skos:exactMatch ?wd .
               FILTER(STRSTARTS(STR(?wd), "http://www.wikidata.org/")) }
    OPTIONAL { ?t mrc:inDecade ?d . }
}
GROUP BY ?label ?start ?wd
ORDER BY ?start
"""
print("tracks per decade:")
print(f"  {'decade':<8} {'tracks':>7}  wikidata")
for row in builder.g.query(q_decade_walk):
    qid = str(row.wd).rsplit("/", 1)[-1] if row.wd else "—"
    print(f"  {str(row.label):<8} {int(row.n_tracks):>7}  {qid}")


tracks per decade:
  decade    tracks  wikidata
  1950s          4  Q36297
  1950s          4  Q67138440
  1960s        128  Q25383554
  1960s        128  Q35724
  1970s        318  Q35014
  1970s        318  Q4350767
  1980s        610  Q34644
  1990s       1536  Q34653
  2000s       2108  Q35024
  2010s         64  Q19022
  1950s          4  Q36297
  1950s          4  Q67138440
  1960s        128  Q25383554
  1960s        128  Q35724
  1970s        318  Q35014
  1970s        318  Q4350767
  1980s        610  Q34644
  1990s       1536  Q34653
  2000s       2108  Q35024
  2010s         64  Q19022


## 8 · Users + listening interactions

Now we fold the KG-restricted Echo Nest taste profile (`taste`) into the graph. Each `(user_id, song_id, play_count)` row becomes a **blank-node listening event** hanging off the user — no per-interaction URI to mint, but the `(user, track, count)` triple stays grouped semantically.

We **reuse** what the ontology already provides:

* `mrc:Listener` — already declared `rdfs:subClassOf foaf:Agent`. Every user is typed as one.
* `mrc:listenCount` (xsd:integer) — already in the ontology, attached to the blank node.

…and add three small schema terms (in `mrc:`, consistent with the existing event-based design):

* `mrc:ListeningEvent` — `owl:Class`, `rdfs:subClassOf event:Event`.
* `mrc:hasListeningInteraction` — `mrc:Listener` → `mrc:ListeningEvent`.
* `mrc:onTrack` — `mrc:ListeningEvent` → `mrc:Track`, declared as `rdfs:subPropertyOf event:factor` so the event-ontology participation semantics carry over.

The shape per row is exactly what you sketched:

```turtle
ex:user/<uid> a mrc:Listener, foaf:Agent ;
    foaf:name "<uid>" ;
    mrc:hasListeningInteraction [
        a              mrc:ListeningEvent ;
        mrc:onTrack    ex:track/<track_id> ;
        mrc:listenCount "42"^^xsd:integer
    ] .
```

> **Memory note.** The rich variant adds ~4 triples per taste row. The KG-restricted profile has ~2.85 M rows → ~11 M extra triples held in RAM.
> If the kernel OOMs, either (a) set `simple=True` in the cell below (1 triple/row, ~75 % less RAM for this section), or (b) pre-filter `taste` to the top-N users before passing it in.
>
> **Corrupt TTL?** If a previous run was interrupted mid-save, the populated TTL on disk may be truncated. In that case set `FORCE_REBUILD = True` in the setup cell and re-run from §4.


In [ ]:
from data.kg import add_users_to_graph

# NOTE — memory cost of the rich variant
# ────────────────────────────────────────
# Each (user, song, play_count) row becomes 4 RDF triples (blank-node
# ListeningEvent pattern). With ~2.85 M taste rows that is ~11 M extra
# triples on top of the already-populated graph, which rdflib keeps fully
# in RAM. If the kernel runs out of memory before the fold completes,
# set  simple=True  below to emit 1 direct triple per row instead (saves
# ~75 % of the listening-section RAM), or pre-filter `taste` to keep only
# the top-N most active users.
#
# NOTE — no checkpoint_every
# ──────────────────────────
# Mid-fold checkpoints call builder.save(), which serialises the *entire*
# graph to Turtle. For a graph this large that pre-processes every triple
# and takes many minutes — don't do it inside the fold. We save once at
# the end instead.

listening_counts = add_users_to_graph(
    builder, taste,
    merged=merged,            # builds the song_id → track_id lookup
    batch_size=100_000,       # rows per inner chunk (tqdm cadence only)
    checkpoint_every=None,    # no mid-fold saves — save once at the end
    verbose=True,
)
print("\nlistening enrichment :", listening_counts)
print("graph stats          :", builder.stats())

builder.save()
print(f"\nupdated {ONTO_OUT}  ({ONTO_OUT.stat().st_size / 1024:,.1f} KiB)")


listening:   9%|▉         | 250k/2.85M [00:27<04:51, 8.93krow/s, interactions=250000, listeners=25088, orphan=0]



KeyboardInterrupt: 

In [ ]:
# 8.1 — sanity-check: top listeners + top tracks by play count.
q_top_listeners = """
PREFIX mrc: <http://purl.org/ontology/mrc/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?u (COUNT(?ev) AS ?n_tracks) (SUM(xsd:integer(?c)) AS ?total_plays)
WHERE {
    ?u a mrc:Listener ;
       mrc:hasListeningInteraction ?ev .
    ?ev mrc:listenCount ?c .
}
GROUP BY ?u
ORDER BY DESC(?total_plays)
LIMIT 5
"""
print("top 5 listeners by total play count:")
for row in builder.g.query(q_top_listeners):
    uid = str(row.u).rsplit("/", 1)[-1]
    print(f"  {uid:<45} tracks={int(row.n_tracks):>4}  plays={int(row.total_plays):>6}")

q_top_tracks = """
PREFIX mrc:    <http://purl.org/ontology/mrc/>
PREFIX dcterms:<http://purl.org/dc/terms/>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
SELECT ?title (COUNT(DISTINCT ?u) AS ?n_users) (SUM(xsd:integer(?c)) AS ?total_plays)
WHERE {
    ?u a mrc:Listener ;
       mrc:hasListeningInteraction ?ev .
    ?ev mrc:onTrack ?t ;
        mrc:listenCount ?c .
    ?t dcterms:title ?title .
}
GROUP BY ?title
ORDER BY DESC(?total_plays)
LIMIT 5
"""
print("\ntop 5 tracks by total play count:")
for row in builder.g.query(q_top_tracks):
    print(f"  {str(row.title)[:50]:<50} users={int(row.n_users):>4}  plays={int(row.total_plays):>6}")


## 9 · Backfill labels for the new `mrc:ElementOfMusic` upper layer

The TTL ontology now declares an `mrc:ElementOfMusic` top concept with `mrc:Genre`,
`mrc:Key` (+ 12 chromatic key individuals), `mrc:Mode` / `mrc:MajorMode` / `mrc:MinorMode`
(linked by `wdt:P13044` *characteristic of* and `wdt:P461` *opposite of*), and
`mrc:TempoClass` (+ 8 tempo individuals) all anchored to Wikidata via `skos:exactMatch`.

To avoid creating a new module, we **reuse** the already-imported
`fetch_qid_metadata` (parallel, cached) + `_attach_qid_metadata` helpers to pull
English `skos:prefLabel`, `skos:definition`, and `skos:altLabel` for the 13 new
QIDs and hang them on the same `wd:Q…` nodes that the TTL just declared.


In [ ]:
# 9.1 — fetch English label/description/aliases for the new QIDs and attach
#       them in-place onto the Wikidata anchor nodes already declared in TTL.
from rdflib import URIRef
from data.kg import fetch_qid_metadata
from data.kg.wikidata_mapping import _attach_qid_metadata

WD_PREFIX        = "http://www.wikidata.org/entity/"
ELEMENTS_SCHEME  = URIRef("http://purl.org/ontology/mrc/scheme/ElementsOfMusic")

# QID → human-readable fallback label (used if Wikidata returns no label).
ELEMENT_QIDS: dict[str, str] = {
    # top-level + class-anchors
    "Q11696608": "element of music",
    "Q188451":   "music genre",
    "Q534932":   "key",
    "Q58795659": "major mode",
    "Q12827391": "minor mode",
    "Q189214":   "tempo",
    # tempo individuals
    "Q2081524":  "Allegro",
    "Q2096748":  "Presto",
    "Q6522688":  "Prestissimo",
    "Q3736203":  "Moderato",
    "Q2499193":  "Andante",
    "Q482153":   "Adagio",
    "Q1805930":  "Largo",
    "Q3730835":  "Grave",
}

elem_meta = fetch_qid_metadata(
    list(ELEMENT_QIDS),
    cache_path=WD_QID_META_PQ,
    force_refresh=FORCE_REBUILD,
    parallel=4,
    batch_size=50,
)

attached = 0
for qid, fallback in ELEMENT_QIDS.items():
    md = elem_meta.get(qid) or {}
    _attach_qid_metadata(
        builder.g,
        URIRef(WD_PREFIX + qid),
        ELEMENTS_SCHEME,
        qid,
        md,
        fallback,
    )
    if md.get("label"):
        attached += 1

builder.save()
print(f"backfilled labels/descriptions/aliases for {attached}/{len(ELEMENT_QIDS)} QIDs")
print(f"updated {ONTO_OUT}  ({ONTO_OUT.stat().st_size / 1024:,.1f} KiB)")


## 10 · Build the *simple* KG variant (no blank nodes)

The graph above is the **rich** variant: per-edge genre weights via blank-node
`mrc:GenreAssociation`, key/mode + confidence on `mo:Performance`, and
listening events as blank-node `mrc:ListeningEvent` carrying `mrc:listenCount`.

For the baseline DL pipeline we also want a **flatter** snapshot with no blank
nodes and no per-edge metadata:

* `mrc:Artist  mrc:hasGenre  mrc:Genre`  — direct, unweighted.
* `mrc:Track   mrc:hasKey    mrc:Key/<x>` and  `mrc:Track  mrc:hasMode  mrc:MajorMode|MinorMode` — no confidence.
* `mrc:Listener  mrc:listenedTo  mrc:Track` — no count, no event node.

We get this by re-running the same pipeline with `KGBuilder(simple=True)` and
serializing to a separate `_simple.ttl`. Both variants share the same
`MusicRecSyst.ttl` upper layer and the same per-track Wikidata-enriched
genre / instrument / decade nodes, so they are 1-to-1 comparable.


In [ ]:
# 10.1 — build & save the simple variant.
from data.kg import KGBuilder, add_users_to_graph

ONTO_OUT_SIMPLE = PROJECT_ROOT / "notebooks" / "MusicRecSyst_populated_simple.ttl"

builder_simple = KGBuilder(
    base_ttl=ONTO_BASE,
    out_ttl=ONTO_OUT_SIMPLE,
    overwrite_copy=True,
    simple=True,
)
builder_simple.add_tempo_class_individuals()

simple_counts = builder_simple.populate_from_dataframe(
    merged, max_rows=None, verbose=True,
)
print("\nminted (simple variant):", simple_counts)

# Same listening rows, but as direct user→track edges (no event node, no count).
# simple=True means 1 triple per row instead of 4 → much lower RAM pressure.
listen_simple = add_users_to_graph(
    builder_simple, taste,
    merged=merged,
    simple=True,
    batch_size=100_000,
    checkpoint_every=None,
    verbose=True,
)
print("listening (simple)      :", listen_simple)

print("\ngraph stats (simple)    :", builder_simple.stats())
builder_simple.save()
print(f"wrote {ONTO_OUT_SIMPLE}  "
      f"({ONTO_OUT_SIMPLE.stat().st_size / 1024:,.1f} KiB)")


In [ ]:
# 10.2 — sanity-check key/mode anchoring + variant differences.
q_key_mode = """
PREFIX mrc: <http://purl.org/ontology/mrc/>
PREFIX skos:<http://www.w3.org/2004/02/skos/core#>
SELECT ?keyLabel ?modeLabel (COUNT(*) AS ?n)
WHERE {
  ?s mrc:hasKey  ?k ;
     mrc:hasMode ?m .
  OPTIONAL { ?k skos:prefLabel ?keyLabel  . FILTER(LANG(?keyLabel)  = "en") }
  OPTIONAL { ?m skos:prefLabel ?modeLabel . FILTER(LANG(?modeLabel) = "en") }
}
GROUP BY ?keyLabel ?modeLabel
ORDER BY DESC(?n)
LIMIT 8
"""

print("RICH variant — top (key, mode) pairs (attached to mo:Performance):")
for row in builder.g.query(q_key_mode):
    print(f"  {str(row[0] or '?'):<6} {str(row[1] or '?'):<8} {int(row[2]):>5}")

print("\nSIMPLE variant — top (key, mode) pairs (attached directly to Track):")
for row in builder_simple.g.query(q_key_mode):
    print(f"  {str(row[0] or '?'):<6} {str(row[1] or '?'):<8} {int(row[2]):>5}")

# Confirm the simple variant has zero blank nodes for genres / listening.
from rdflib import BNode
n_bnodes_rich   = sum(1 for s in builder.g.subjects()        if isinstance(s, BNode))
n_bnodes_simple = sum(1 for s in builder_simple.g.subjects() if isinstance(s, BNode))
print(f"\nblank-node subject count — rich: {n_bnodes_rich:,}   simple: {n_bnodes_simple:,}")
